# 04 · matplotlib and quick charts

After this notebook you can go from a DataFrame to the right chart quickly. That covers the figure/axes model, every basic chart type, annotations and log scales, a Power BI-style dashboard, and the Power BI *Python visual* convention. You will also be able to draw the charts an AI engineer is expected to produce: loss curves, a confusion matrix, a latency histogram with p95, an embedding PCA scatter, and token usage over time.

**Time:** ~40 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · pandas essentials](02_pandas_essentials.ipynb)

## Why this matters in interviews

- Take-homes and system-design follow-ups end with "show me". A loss curve that reveals overfitting, or a latency histogram with the p95 marked, communicates more than a table.
- "Fast chart creation" (Power BI, dashboards) is a common ask for AI engineers who work with analysts. Knowing the Python-visual contract, including the `dataset` DataFrame and its silent de-duplication, is a practical edge.
- Choosing the right chart for the question, and avoiding misleading ones such as dual axes and rainbow colour maps, is a signal of judgment.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `po23` | What would you put on a monitoring dashboard for a GenAI system? |
| `in31` | What metrics would you put on an inference dashboard? |
| `pd41` | What is the difference between p50, p95 and p99, and which do you optimise? |
| `ev17` | How do you evaluate a classification task done by an LLM? |
| `tn28` | What is overfitting in the context of fine-tuning an LLM, and how do you spot it? |
| `po32` | Token usage doubled overnight with no deploy. How do you find the cause? |

A small **house style** first: a fixed colour order (validated for colour-blind readers, blue then orange then aqua), light solid gridlines, no top or right spines. Set it once with `rcParams` and every chart in the notebook is consistent.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch

BLUE, ORANGE, AQUA, YELLOW, GRAY, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b5b3ab", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW, "#e87ba4", "#008300", "#4a3aa7", "#e34948"]),
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.titlesize": 11, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False,
})
rng = np.random.default_rng(0)
TMP = Path(tempfile.mkdtemp())
import matplotlib; print("matplotlib", matplotlib.__version__)

## 1. The figure/axes model

**In plain English:** a **Figure** is the canvas, and an **Axes** is one plot on it, with its own x-axis, y-axis, title and legend. `fig, ax = plt.subplots()` gives you both. Then *call methods on `ax`*: `ax.plot`, `ax.set_title`, `ax.legend`. The `plt.plot(...)` style draws on whatever axes is "current". It is fine for a one-off, but it gets confusing as soon as there are two plots. Write the explicit `ax.` style.

In [ ]:
x = np.linspace(0, 10, 50)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(x, np.sqrt(x), marker="o", ms=3, label="series A")
ax.plot(x, np.log1p(x), label="series B")
ax.set_title("Title (ax.set_title)"); ax.set_xlabel("x label (ax.set_xlabel)"); ax.set_ylabel("y label")
ax.legend(loc="lower right")
callout = dict(fontsize=9, color=ORANGE, fontweight="bold", xycoords="axes fraction", textcoords="axes fraction",
               arrowprops=dict(arrowstyle="->", color=ORANGE))
ax.annotate("Axes = one plot area", xy=(0.3, 0.55), xytext=(0.08, 0.8), **callout)
ax.annotate("legend (ax.legend)", xy=(0.86, 0.2), xytext=(0.62, 0.42), **callout)
ax.annotate("tick labels", xy=(0.37, -0.06), xytext=(0.45, 0.1), **callout)
fig.text(0.01, 0.97, "Figure = the whole canvas (fig)", fontsize=9, color=ORANGE, fontweight="bold", va="top")
fig.patch.set_edgecolor(ORANGE); fig.patch.set_linewidth(2)
plt.show()
assert len(fig.axes) == 1 and fig.axes[0] is ax

## 2. The basic chart types, in a subplot grid

`plt.subplots(rows, cols)` returns an array of axes. Loop over it or index it (`axes[0, 1]`). Each panel below names the question that chart answers.

In [ ]:
cats, vals = ["A", "B", "C", "D"], [23, 17, 35, 12]
fig, axes = plt.subplots(2, 3, figsize=(11, 4.5))
axes[0, 0].plot(np.arange(12), np.cumsum(rng.normal(1, 2, 12)), marker="o", ms=3)
axes[0, 0].set_title("line: change over time")
axes[0, 1].bar(cats, vals, color=BLUE, width=0.6)
axes[0, 1].set_title("bar: compare categories")
axes[0, 2].barh(["long label one", "label two", "three"], [5, 9, 3], color=BLUE, height=0.55)
axes[0, 2].set_title("barh: long labels, rankings")
axes[1, 0].hist(rng.normal(0, 1, 1_000), bins=30, color=BLUE)
axes[1, 0].set_title("hist: a distribution")
sx = rng.normal(0, 1, 150)
axes[1, 1].scatter(sx, 2 * sx + rng.normal(0, 1, 150), s=10, alpha=0.7)
axes[1, 1].set_title("scatter: relationship of two numbers")
axes[1, 2].boxplot([rng.normal(m, s, 200) for m, s in [(0, 1), (1, 0.5), (0.5, 2)]], tick_labels=["x", "y", "z"])
axes[1, 2].set_title("box: distributions side by side")
plt.tight_layout(); plt.show()

### `imshow` heatmaps

A heatmap shows a matrix. Pick the colour map by the job the colour does. A **sequential** map (one hue, light to dark, such as `Blues`) is for magnitudes like counts or similarity, and a **diverging** map (two hues with a neutral middle, such as `RdBu`) is for signed values like correlations. Never use a rainbow map. Always add a colour bar, and annotate the cells when there are few of them.

In [ ]:
metrics = pd.DataFrame({
    "latency": rng.lognormal(7, 0.3, 300),
})
metrics["tokens_out"] = metrics["latency"] * 0.2 + rng.normal(0, 40, 300)
metrics["cost"] = metrics["tokens_out"] * 0.0016 + rng.normal(0, 0.05, 300)
metrics["user_rating"] = -0.004 * metrics["latency"] + rng.normal(8, 1, 300)
metrics["retrieval_hits"] = rng.integers(0, 6, 300)
corr = metrics.corr()

fig, ax = plt.subplots(figsize=(6.2, 4.4))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=30, ha="right"); ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else INK)
ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.85, label="correlation")
ax.set_title("Correlation heatmap: a diverging map, since the sign matters")
plt.tight_layout(); plt.show()
assert np.allclose(np.diag(corr), 1)

## 3. Annotations, log scales, twin axes

- `ax.annotate(text, xy=point, xytext=label_position, arrowprops=...)` points at the one value that matters. Label selectively, not every point.
- **Log scales** are for data that spans orders of magnitude, such as token frequencies, latencies or loss. A straight line on log-log axes means a power law.

In [ ]:
ranks = np.arange(1, 2_001)
freq = 50_000 / ranks ** 1.05 * rng.lognormal(0, 0.1, ranks.size)      # Zipf-like token frequencies

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(ranks, freq, color=BLUE)
axes[0].set_title("Linear axes: everything is squashed into a corner")
axes[1].loglog(ranks, freq, color=BLUE)
axes[1].annotate("'the', ',', '.' ... a few tokens\nare most of the text", xy=(1.5, freq[0]), xytext=(20, 3_000),
                 fontsize=8, color=MUTED, arrowprops=dict(arrowstyle="->", color=MUTED))
axes[1].set_title("Log-log axes: a straight line = a power law (Zipf)")
for a in axes:
    a.set_xlabel("token rank"); a.set_ylabel("count")
plt.tight_layout(); plt.show()

**Twin axes** (`ax.twinx()`) put a second y-scale on the same plot. You need to know the API, because colleagues use it, but use it carefully. The alignment of the two scales is arbitrary, so the chart can *suggest* a correlation just by how you choose the ranges. The honest alternative is two panels that share the x-axis (`sharex=True`):

In [ ]:
hours = pd.date_range("2026-09-01", periods=24, freq="h")
requests = (400 + 300 * np.sin((np.arange(24) - 8) / 24 * 2 * np.pi) + rng.normal(0, 30, 24)).clip(50)
p95_ms = 1_200 + 0.9 * requests + rng.normal(0, 60, 24)

fig = plt.figure(figsize=(11, 3.8))
left = fig.add_subplot(1, 2, 1)
left.bar(hours, requests, width=0.03, color=GRAY, label="requests / h")
right_axis = left.twinx()                                  # the second y-scale
right_axis.plot(hours, p95_ms, color=ORANGE, marker="o", ms=3, label="p95 ms")
left.set_title("twinx(): two scales, one plot (use with care)")
left.set_ylabel("requests / hour"); right_axis.set_ylabel("p95 latency (ms)", color=ORANGE)
right_axis.spines["right"].set_visible(True); right_axis.grid(False)
left.tick_params(axis="x", labelsize=7, rotation=30)

top = fig.add_subplot(2, 2, 2); bottom = fig.add_subplot(2, 2, 4, sharex=top)
top.bar(hours, requests, width=0.03, color=BLUE); top.set_ylabel("req / h")
top.set_title("Better: two panels sharing the x-axis"); top.tick_params(labelbottom=False)
bottom.plot(hours, p95_ms, color=BLUE, marker="o", ms=3); bottom.set_ylabel("p95 ms")
bottom.tick_params(axis="x", labelsize=7, rotation=30)
plt.tight_layout(); plt.show()

## 4. Saving figures

`fig.savefig(path, dpi=..., bbox_inches="tight")`. PNG is for docs and slides. SVG or PDF are vector formats, and they stay sharp in reports. In a loop that makes many figures, call `plt.close(fig)` so they don't pile up in memory. The pixel size is `figsize × dpi`, and `bbox_inches="tight"` trims the white margins.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot([1, 2, 3], [3, 1, 2]); ax.set_title("saved to disk")
fig.savefig(TMP / "chart.png", dpi=100)                              # exact size: 400 x 250 px
fig.savefig(TMP / "chart_tight.png", dpi=150, bbox_inches="tight")
fig.savefig(TMP / "chart.svg")
plt.close(fig)                                                       # not shown inline: we only saved it
height, width = plt.imread(TMP / "chart.png").shape[:2]
print(f"chart.png {width}x{height} px | files: {sorted(p.name for p in TMP.iterdir())}")
assert (width, height) == (400, 250) and (TMP / "chart.svg").stat().st_size > 0

## 5. pandas `.plot` shortcuts

`df.plot(...)`, `df.plot.bar()`, `series.plot.hist()` and `df.plot.scatter()` are one-liners over matplotlib. Each returns an `Axes`, so you keep customising with the same `ax.` methods, and you can pass `ax=` to draw into your own grid. The pattern is always **aggregate first, then plot**.

In [ ]:
days = pd.date_range("2026-01-01", "2026-06-30", freq="D")
n = 900
sales = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "date": rng.choice(days, n),
    "region": rng.choice(["North", "South", "East", "West"], n, p=[0.35, 0.25, 0.25, 0.15]),
    "product": rng.choice(["Basic", "Pro", "Team"], n, p=[0.55, 0.3, 0.15]),
})
sales["revenue"] = sales["product"].map({"Basic": 49, "Pro": 149, "Team": 399}) * rng.integers(1, 4, n)
sales["month"] = sales["date"].dt.to_period("M").astype(str)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
sales.groupby("month")["revenue"].sum().plot(ax=axes[0], marker="o", title="df.plot(): revenue by month")
sales.groupby("region")["revenue"].sum().sort_values().plot.barh(ax=axes[1], color=BLUE, title=".plot.barh(): by region")
sales["revenue"].plot.hist(ax=axes[2], bins=12, color=BLUE, title=".plot.hist(): order values")
axes[0].tick_params(axis="x", rotation=30, labelsize=8)
plt.tight_layout(); plt.show()

## 6. A "Power BI style" 2×2 dashboard

A dashboard is a grid: a KPI row that answers "how are we doing?" in numbers, then charts that answer "where?" and "when?". KPI cards are just text on an axes with the axis turned off. Keep one colour for one series, sort the bars, and give every panel a title that states its question.

In [ ]:
def kpi_cards(ax, cards):
    """Draw up to 4 KPI cards (label, value, sub-line) in a 2x2 layout on one axes."""
    ax.axis("off")
    for k, (label, value, sub) in enumerate(cards):
        x0, y0 = (k % 2) * 0.5, 0.5 - (k // 2) * 0.5
        ax.add_patch(FancyBboxPatch((x0 + 0.02, y0 + 0.04), 0.46, 0.42, boxstyle="round,pad=0.01,rounding_size=0.03",
                                    fc="#f4f7fd", ec="#d5e2f7", transform=ax.transAxes))
        ax.text(x0 + 0.05, y0 + 0.36, label, fontsize=8, color=MUTED, transform=ax.transAxes)
        ax.text(x0 + 0.05, y0 + 0.17, value, fontsize=15, color=INK, fontweight="bold", transform=ax.transAxes)
        ax.text(x0 + 0.05, y0 + 0.07, sub, fontsize=7, color=MUTED, transform=ax.transAxes)

monthly = sales.groupby("month")["revenue"].sum()
growth = monthly.iloc[-1] / monthly.iloc[-2] - 1
cards = [("Revenue (H1)", f"${sales['revenue'].sum() / 1e3:,.0f}k", "Jan-Jun 2026"),
         ("Orders", f"{len(sales):,}", f"{len(sales) / len(days):.1f} per day"),
         ("Avg order value", f"${sales['revenue'].mean():,.0f}", "revenue / orders"),
         ("June vs May", f"{growth:+.1%}", "month-over-month")]

fig, axes = plt.subplots(2, 2, figsize=(11, 4.5))
kpi_cards(axes[0, 0], cards)
axes[0, 1].plot(monthly.index, monthly.values / 1e3, marker="o", color=BLUE)
best = monthly.idxmax()
axes[0, 1].annotate(f"best month: {best}", xy=(best, monthly.max() / 1e3), xytext=(8, 6), textcoords="offset points",
                    ha="left", fontsize=8, color=MUTED)
axes[0, 1].set_title("Revenue by month ($k)"); axes[0, 1].set_ylim(0, monthly.max() / 1e3 * 1.25)
by_region = sales.groupby("region")["revenue"].sum().sort_values()
axes[1, 0].barh(by_region.index, by_region.values / 1e3, color=BLUE, height=0.55)
axes[1, 0].set_title("Revenue by region ($k)")
by_product = sales.pivot_table(index="product", values="revenue", aggfunc="sum")["revenue"].sort_values(ascending=False)
axes[1, 1].bar(by_product.index, by_product.values / 1e3, color=BLUE, width=0.55)
axes[1, 1].set_title("Revenue by product ($k)"); axes[1, 1].set_ylim(0, by_product.max() / 1e3 * 1.15)
fig.suptitle("Sales dashboard - H1 2026", fontsize=12, x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()
assert np.isclose(by_region.sum(), sales["revenue"].sum())

## 7. The Power BI Python-visual convention

In Power BI Desktop you add a **Python visual** and drag fields into it. Power BI then runs your script with those fields as a pandas DataFrame called **`dataset`**, after running this preamble, which you cannot edit:

```python
# dataset = pandas.DataFrame(region, product, revenue)
# dataset = dataset.drop_duplicates()
```

Your script draws with matplotlib and ends with `plt.show()`, and Power BI displays the rendered image. What to remember:

| Fact | Consequence |
|---|---|
| rows are **de-duplicated** | identical rows merge and totals shrink, so add an ID field |
| the script sees **only `dataset`** | no notebook variables, and it must import what it uses |
| the output is a **static image** | slicers and filters re-run it, but you cannot click it to cross-filter |
| at most **150,000 rows**, about a **5-minute** timeout | aggregate in Power Query or DAX first when you can |
| needs a local Python with pandas + matplotlib (Desktop) | the Service supports a fixed set of packages |

Below we emulate it. `run_python_visual` builds `dataset` exactly as the preamble does and runs the script with *nothing else* in scope. That is a handy way to test a visual before pasting it into Power BI.

In [ ]:
POWER_BI_SCRIPT = '''
import matplotlib.pyplot as plt
summary = dataset.groupby("region")["revenue"].sum().sort_values()
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(summary.index, summary.values, color="#2a78d6", height=0.55)
ax.set_title("Python visual: revenue by region")
plt.show()
'''

def run_python_visual(script: str, table: pd.DataFrame, fields: list[str]) -> pd.DataFrame:
    dataset = pd.DataFrame(table[fields]).drop_duplicates()      # Power BI's preamble
    exec(script, {"dataset": dataset})                            # the script sees ONLY `dataset`
    return dataset

ds_no_id = run_python_visual(POWER_BI_SCRIPT, sales, ["region", "product", "revenue"])
ds_with_id = run_python_visual(POWER_BI_SCRIPT, sales, ["order_id", "region", "product", "revenue"])
print(f"without an ID field: {len(ds_no_id)} rows, revenue {ds_no_id['revenue'].sum():,}  <- merged duplicates")
print(f"with order_id      : {len(ds_with_id)} rows, revenue {ds_with_id['revenue'].sum():,}  (the true total)")
assert ds_with_id["revenue"].sum() == sales["revenue"].sum() > ds_no_id["revenue"].sum()

try:
    exec("print(len(sales))", {"dataset": ds_with_id})             # a script that leans on notebook state
except NameError as e:
    print("a script that uses notebook variables fails in Power BI too:", e)

## 8. Which chart for which question

| The question | Chart | Avoid |
|---|---|---|
| How does it change over time? | line (many points), column (few periods) | pie, unsorted bars |
| Which category is biggest? | sorted **bar / barh** | pie with many slices, 3-D bars |
| How is it distributed? What's the tail? | **histogram** (+ p50/p95 lines), box plot | a bar chart of the mean only |
| Do two numbers move together? | **scatter** (+ trend line) | dual-axis line chart |
| Part of a whole, 2–5 parts | stacked bar, or a pie at most | a pie with 10 slices |
| A matrix (confusion, similarity, correlation) | **heatmap** with annotations and a colour bar | rainbow colour maps |
| One headline number | a **KPI card** | a one-bar chart |
| Many groups, the same chart | **small multiples** (a subplot grid, shared axes) | 8 lines in one tangle |
| High-dimensional points (embeddings) | 2-D **PCA / UMAP scatter** coloured by label | reading distances literally |

## 9. The AI engineer's standard charts

### Loss curves (`tn28`)

Train and validation loss per epoch on the same axes. The moment validation turns up while training keeps falling is the overfitting point. Mark the best epoch, because that is the checkpoint you keep (early stopping).

In [ ]:
epochs = np.arange(1, 31)
train_loss = 2.2 * np.exp(-epochs / 6) + 0.25 + rng.normal(0, 0.015, epochs.size)
val_loss = 2.2 * np.exp(-epochs / 6) + 0.42 + 0.0035 * np.clip(epochs - 11, 0, None) ** 2 + rng.normal(0, 0.02, epochs.size)
best_epoch = int(epochs[np.argmin(val_loss)])

fig, ax = plt.subplots(figsize=(8.5, 3.4))
ax.plot(epochs, train_loss, color=BLUE, label="train loss")
ax.plot(epochs, val_loss, color=ORANGE, label="validation loss")
ax.axvline(best_epoch, color=GRAY, lw=1)
ax.axvspan(best_epoch, epochs[-1], color=ORANGE, alpha=0.06, lw=0)
ax.annotate(f"best checkpoint: epoch {best_epoch}", xy=(best_epoch, val_loss.min()), xytext=(best_epoch + 2, 1.6),
            fontsize=9, color=MUTED, arrowprops=dict(arrowstyle="->", color=MUTED))
ax.text(best_epoch + 8, 0.95, "overfitting: validation rises,\ntrain keeps falling", fontsize=8, color=MUTED)
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("Loss curves with the early-stopping point marked")
ax.legend(); plt.show()
assert train_loss[-1] < train_loss[best_epoch - 1] and val_loss[-1] > val_loss.min()

### Confusion matrix heatmap (`ev17`)

For an LLM classifier (a router, an intent or sentiment labeller), accuracy hides *which* mistakes are made. The confusion matrix shows them: rows are the truth, columns the prediction, and the off-diagonal cells are the errors to fix. Normalise each row to read **recall** per class.

In [ ]:
classes = ["billing", "technical", "account"]
truth = rng.choice(3, 300, p=[0.4, 0.35, 0.25])
confuse = np.array([[0.90, 0.04, 0.06],        # billing is rarely confused ...
                    [0.05, 0.80, 0.15],        # ... but technical often lands in account
                    [0.08, 0.12, 0.80]])
pred = np.array([rng.choice(3, p=confuse[t]) for t in truth])
cm = np.zeros((3, 3), dtype=int)
np.add.at(cm, (truth, pred), 1)                # count (true, predicted) pairs
assert cm.sum() == 300 and np.trace(cm) == (truth == pred).sum()

fig, ax = plt.subplots(figsize=(5.2, 4))
im = ax.imshow(cm, cmap="Blues")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i, j]}\n{cm[i, j] / cm[i].sum():.0%}", ha="center", va="center", fontsize=9,
                color="white" if cm[i, j] > cm.max() / 2 else INK)
ax.set_xticks(range(3), classes); ax.set_yticks(range(3), classes); ax.grid(False)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.colorbar(im, ax=ax, shrink=0.85, label="count")
ax.set_title(f"Confusion matrix (accuracy {np.trace(cm) / cm.sum():.0%})")
plt.tight_layout(); plt.show()

### Latency histogram with p50 / p95 / p99 (`pd41`)

In [ ]:
lat = rng.lognormal(np.log(850), 0.35, 3_000)
slow = rng.random(lat.size) < 0.03
lat[slow] += rng.uniform(2_000, 5_000, slow.sum())            # retries and cold starts
p50, p95, p99 = np.percentile(lat, [50, 95, 99])

fig, ax = plt.subplots(figsize=(9.5, 3.4))
ax.hist(lat, bins=np.linspace(0, 6_500, 100), color=BLUE)
for value, name, color in [(p50, "p50", AQUA), (p95, "p95", ORANGE), (p99, "p99", "#e34948")]:
    ax.axvline(value, color=color, lw=1.8)
    ax.text(value + 60, ax.get_ylim()[1] * 0.9, f"{name}\n{value:,.0f} ms", fontsize=8, color=MUTED, va="top")
ax.set_xlabel("latency (ms)"); ax.set_ylabel("requests")
ax.set_title(f"Latency: the median is {p50:,.0f} ms but 1 in 20 requests takes over {p95:,.0f} ms")
plt.show()
assert p50 < p95 < p99

### 2-D PCA scatter of embeddings

Embeddings have hundreds of dimensions. **PCA** projects them onto the two directions of greatest variance, so you can *see* whether topics separate. It is a sanity check for an embedding model or a clustering, not a precise measurement: distances in 2-D are distorted. PCA is a few lines of NumPy with an SVD:

In [ ]:
topics = ["refunds", "passwords", "shipping"]
centers = rng.normal(0, 1, (3, 64))
X_emb = np.vstack([c + rng.normal(0, 0.8, (60, 64)) for c in centers])
y_emb = np.repeat(np.arange(3), 60)

Xc = X_emb - X_emb.mean(axis=0)                       # PCA = SVD of the centred data
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
coords = Xc @ Vt[:2].T                                # project onto the top-2 principal directions
explained = S ** 2 / (S ** 2).sum()
assert np.all(np.diff(explained) <= 0) and np.isclose(explained.sum(), 1)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for k, (name, color) in enumerate(zip(topics, [BLUE, ORANGE, AQUA])):
    pts = coords[y_emb == k]
    ax.scatter(pts[:, 0], pts[:, 1], s=16, color=color, alpha=0.8, edgecolor="white", linewidth=0.5, label=name)
    ax.text(pts[:, 0].mean(), pts[:, 1].max() + 0.6, name, fontsize=9, fontweight="bold", color=INK, ha="center")
ax.set_xlabel(f"PC1 ({explained[0]:.0%} of variance)"); ax.set_ylabel(f"PC2 ({explained[1]:.0%} of variance)")
ax.set_title("180 embeddings (64-d) projected to 2-D with PCA")
ax.margins(y=0.15); ax.legend(fontsize=8, loc="center"); plt.show()

### Token usage over time (`po32`)

Tokens drive cost, so plot them over time and split prompt from completion. A stacked area shows the total and its parts. When a spike appears, the next step is the pandas drill-down (by user, model and feature) from [02 · pandas essentials](02_pandas_essentials.ipynb).

In [ ]:
t = pd.date_range("2026-09-01", periods=72, freq="h")
daily_cycle = 1 + 0.6 * np.sin((t.hour.to_numpy() - 9) / 24 * 2 * np.pi)
prompt_tok = (40_000 * daily_cycle * rng.lognormal(0, 0.1, 72)).round()
completion_tok = (9_000 * daily_cycle * rng.lognormal(0, 0.1, 72)).round()
spike = t >= "2026-09-03 02:00"
prompt_tok[spike] *= 2.1                                   # something doubled the prompts on day 3

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.stackplot(t, prompt_tok / 1e3, completion_tok / 1e3, colors=[BLUE, ORANGE], alpha=0.9,
             labels=["prompt tokens", "completion tokens"], edgecolor="white", linewidth=0.5)
ax.annotate("prompt tokens doubled overnight\n(completions didn't: suspect context growth)",
            xy=(t[spike][4], (prompt_tok[spike][4] + completion_tok[spike][4]) / 1e3), xytext=(t[20], 125),
            fontsize=8, color=MUTED, arrowprops=dict(arrowstyle="->", color=MUTED))
ax.set_ylabel("thousand tokens / hour"); ax.set_title("Token usage per hour, stacked")
ax.legend(loc="upper left", fontsize=8); ax.tick_params(axis="x", labelsize=8)
plt.show()
ratio = prompt_tok[spike].sum() / completion_tok[spike].sum() / (prompt_tok[~spike].sum() / completion_tok[~spike].sum())
print(f"prompt:completion ratio went up {ratio:.1f}x after the spike")
assert ratio > 1.8

## Try it yourself

**1.** Redraw the latency histogram with the x-axis on a **log scale**, a shaded region beyond a 2,000 ms SLO, and the share of requests that breach it in the title.

In [ ]:
# Solution — try it yourself first, then run this
slo = 2_000
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.hist(lat, bins=np.logspace(np.log10(lat.min()), np.log10(lat.max()), 80), color=BLUE)
ax.set_xscale("log")
ax.axvspan(slo, lat.max(), color=ORANGE, alpha=0.12, lw=0)
ax.axvline(slo, color=ORANGE, lw=1.5)
ax.set_xlabel("latency (ms, log scale)"); ax.set_ylabel("requests")
ax.set_title(f"{(lat > slo).mean():.1%} of requests breach the {slo:,} ms SLO")
plt.show()

**2.** From `cm`, compute **recall** (the diagonal over each row sum) and **precision** (the diagonal over each column sum) for every class, and say which class the classifier over-predicts.

In [ ]:
# Solution — try it yourself first, then run this
recall = np.diag(cm) / cm.sum(axis=1)
precision = np.diag(cm) / cm.sum(axis=0)
for c, r, p in zip(classes, recall, precision):
    print(f"{c:10} recall {r:.2f} | precision {p:.2f}")
print("over-predicted (lowest precision):", classes[int(np.argmin(precision))])
assert np.isclose((recall * cm.sum(axis=1)).sum() / cm.sum(), np.trace(cm) / cm.sum())

**3.** Small multiples: generate latencies for three models over three days, and draw a 1×3 grid (one panel per model, shared y-axis) of p95 per day.

In [ ]:
# Solution — try it yourself first, then run this
median = {"gpt-4.1-mini": 900, "gpt-4.1": 1700, "qwen2.5:3b": 1300}
runs = pd.DataFrame([(m, d, rng.lognormal(np.log(med * (1.25 if (m == "gpt-4.1" and d == "Wed") else 1)), 0.35))
                     for m, med in median.items() for d in ["Mon", "Tue", "Wed"] for _ in range(200)],
                    columns=["model", "day", "latency_ms"])
p95_grid = runs.groupby(["model", "day"])["latency_ms"].quantile(0.95).unstack()[["Mon", "Tue", "Wed"]]
fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), sharey=True)
for ax, (model_name, row) in zip(axes, p95_grid.iterrows()):
    ax.bar(row.index, row.values, color=BLUE, width=0.55)
    ax.set_title(model_name)
axes[0].set_ylabel("p95 latency (ms)")
plt.tight_layout(); plt.show()

**4.** Save the sales dashboard as a 150-dpi PNG in the temp folder and check its pixel size (`figsize × dpi` without `bbox_inches="tight"`).

In [ ]:
# Solution — try it yourself first, then run this
fig, axes = plt.subplots(2, 2, figsize=(11, 4.5))
kpi_cards(axes[0, 0], cards)
axes[0, 1].plot(monthly.index, monthly.values, marker="o")
axes[1, 0].barh(by_region.index, by_region.values, color=BLUE)
axes[1, 1].bar(by_product.index, by_product.values, color=BLUE)
fig.savefig(TMP / "dashboard.png", dpi=150)
plt.close(fig)
h_px, w_px = plt.imread(TMP / "dashboard.png").shape[:2]
print(f"dashboard.png: {w_px} x {h_px} px")
assert (w_px, h_px) == (1650, 675)

## Say it in an interview

- **"What goes on a GenAI monitoring dashboard?"** (`po23`, `in31`) Use a KPI row plus time series. Traffic (requests per minute), **p50/p95/p99 latency** and time to first token, error and timeout rate by type (429s, 5xx), **tokens and cost** per hour split by model and feature, cache hit rate, and quality signals (thumbs-down rate, eval scores, guardrail triggers). Every panel should answer a question and have an alert threshold behind it.
- **"How do you show a classifier's quality?"** (`ev17`) Show a confusion matrix, with row-normalised recall and precision per class, not just accuracy. The off-diagonal cells tell you which labels or prompts to fix.
- **"How do you spot overfitting?"** (`tn28`) Plot train and validation loss per epoch on one chart. The gap opening, with validation turning up, is the sign. Keep the checkpoint at the validation minimum.
- **"Why p95 and not the mean?"** (`pd41`) A histogram shows it immediately: the mean sits in the body, and the tail is where users suffer.
- **Power BI Python visuals:** your script gets a de-duplicated `dataset` DataFrame, so add an ID field. You draw with matplotlib and end with `plt.show()`. The output is a static image, with a 150k-row cap and roughly a 5-minute timeout. Use native visuals when you need interactivity.
- **Chart judgment:** sort your bars, use one colour per series (a highlight colour plus gray for "one thing matters"), sequential colour maps for magnitude and diverging ones for signed values, and never a rainbow. Avoid dual axes: two panels with `sharex=True` are honest. Direct-label the value that matters instead of labelling every point.
- **matplotlib mechanics:** use `fig, ax = plt.subplots()` and the `ax.` methods, `tight_layout()`, `savefig(dpi=, bbox_inches="tight")`, and `plt.close(fig)` in loops. Pixel size is figsize × dpi.

## Next

- Back to [02 · pandas essentials](02_pandas_essentials.ipynb) for the aggregation behind every chart here, or on to [03 · PyTorch essentials](03_pytorch_essentials.ipynb).
- Where these charts are used: [metrics, confusion matrix, ROC & PR](../05_machine_learning/04_metrics_confusion_matrix_roc_pr.ipynb) · [clustering & PCA](../05_machine_learning/08_clustering_kmeans_and_pca.ipynb) · [training loops & LR schedules](../06_deep_learning/02_training_loop_optimizers_lr_schedules.ipynb) · [latency percentiles & cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) · [observability & tracing](../16_production/08_observability_logging_and_tracing.ipynb)
- The quick version: [quick interview topics, #2 Power BI](../00_start_here/01_quick_interview_topics.ipynb)
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html) · [interview bank](../../ai_interview_prep/index.html)